# Required libraries

In [ ]:
import time
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns
import tensorflow as tf
from tensorflow import keras
#from keras.optimizers import Adam
from keras.losses import Loss
from keras.initializers import HeNormal, HeUniform, GlorotUniform
from keras.utils import to_categorical
from joblib import Parallel, delayed, parallel_backend
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from tqdm.auto import tqdm


In [ ]:
class TQDMProgressCallback(keras.callbacks.Callback):
    """
    Keras callback that wraps model.fit() with a rich tqdm progress bar.
    Reports: experiment label, start time, per-epoch timing, ETA, end time, total elapsed.
    """
    def __init__(self, desc="Training"):
        super().__init__()
        self.desc = desc

    def on_train_begin(self, logs=None):
        self._start_time = time.time()
        print(f"\n{'='*60}")
        print(f"  [{self.desc}]  Start: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*60}")
        self._pbar = tqdm(
            total=self.params['epochs'],
            desc="  Epochs",
            unit="ep",
            dynamic_ncols=True,
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} ep "
                       "[{elapsed}<{remaining}, {rate_fmt}{postfix}]",
        )
        self._epoch_times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        ep_secs = time.time() - self._epoch_start
        self._epoch_times.append(ep_secs)
        postfix = {k: f"{v:.4f}" for k, v in (logs or {}).items()}
        postfix["s/ep"] = f"{ep_secs:.2f}"
        self._pbar.set_postfix(postfix, refresh=False)
        self._pbar.update(1)

    def on_train_end(self, logs=None):
        self._pbar.close()
        total_secs = time.time() - self._start_time
        avg_ep = sum(self._epoch_times) / len(self._epoch_times) if self._epoch_times else 0
        m, s = divmod(total_secs, 60)
        print(f"\n  [{self.desc}]  End  : {time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"  Total time   : {int(m)}m {s:.1f}s  |  Avg per epoch: {avg_ep:.2f}s")
        print(f"{'='*60}\n")


## GPU visibility

In [ ]:
# To disable GPU usage, uncomment the following lines:
#import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
# print(tf.config.experimental.list_physical_devices())
# gpus = tf.config.list_physical_devices('GPU'); print(gpus)
# tf.config.set_visible_devices([gpus[0]], 'GPU')
# tf.config.get_visible_devices('GPU')

In [ ]:
# To prevent GPU memory fragmentation, you can set a memory limit (adjust as needed):
# gpus = tf.config.get_visible_devices('GPU') # tf.config.list_physical_devices('GPU')
# if gpus:
#     try:
#         for gpu in gpus:
#             #tf.config.experimental.set_memory_growth(gpu, True)
#             tf.config.experimental.set_virtual_device_configuration(gpu, [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=3774)])
#             print(f"Enabled memory growth for GPU: {gpu.name}")
#         logical_gpus = tf.config.list_logical_devices('GPU')
#         print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs configured.")
#     except RuntimeError as e:
#         print(f"RuntimeError during GPU configuration: {e}")
#         print("Ensure GPU memory configuration is set at the very beginning of your script.")
# else:
#     print("No GPU devices found. Running on CPU.")

# Load your dataset

In [ ]:

# ── PART 1: Dataset Config ────────────────────────────────────────────────────
# Set DATASET = 'cifar10' or 'mnist'.  Both are normalised to [0,1] and padded
# to the same 32×32 spatial resolution so one transformer checkpoint handles
# both datasets without any other code changes.
# ─────────────────────────────────────────────────────────────────────────────

DATASET = 'cifar10'   # ← change to 'mnist' to switch dataset

if DATASET == 'cifar10':
    (X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
    IMG_H, IMG_W, IN_CHANNELS, NUM_CLASSES = 32, 32, 3, 10

elif DATASET == 'mnist':
    (X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
    # Pad 28×28 → 32×32 (2-pixel border) so patch_size=8 tiles evenly,
    # then add an explicit channel dimension.
    X_train = np.pad(X_train, ((0,0),(2,2),(2,2)), mode='constant')[..., np.newaxis]
    X_test  = np.pad(X_test,  ((0,0),(2,2),(2,2)), mode='constant')[..., np.newaxis]
    IMG_H, IMG_W, IN_CHANNELS, NUM_CLASSES = 32, 32, 1, 10

else:
    raise ValueError(f"Unknown DATASET '{DATASET}'.  Use 'cifar10' or 'mnist'.")

y_train = y_train.flatten()
y_test  = y_test.flatten()

X_train = X_train / 255.0   # Normalise pixel values to [0, 1]
X_test  = X_test  / 255.0

print(f"Dataset : {DATASET.upper()} | "
      f"Train {X_train.shape} | Test {X_test.shape} | Classes {NUM_CLASSES}")


In [ ]:
# Contamination introducer, if you want to artificially corrupt the labels to test robustness
# def corrupt_labels(y_train, prob, num_classes=10, seed=None):
#     if seed is not None:
#         np.random.seed(seed)

#     y_corrupted = y_train.copy()
#     n = len(y_train)
#     # Decide which labels to corrupt
#     corrupt_mask = np.random.rand(n) < prob
#     # For each label to corrupt, choose a new label different from the original
#     for i in np.where(corrupt_mask)[0]:
#         original_label = y_train[i]
#         # possible new labels excluding the original
#         new_labels = list(range(num_classes))
#         new_labels.remove(original_label)
#         # randomly pick a new label
#         y_corrupted[i] = np.random.choice(new_labels)

#     return y_corrupted

# Different robust loss functions, as Python objects

In [ ]:
# SD-loss implementation
class SDIV(Loss):
    def __init__(self, beta, lam, trim_ratio): # beta and lambda are the SD tuning parameter. Set trim_ratio to 0 to disable trimming, as done in our paper.
        super().__init__()
        self.beta = float(beta)
        self.lam = float(lam)
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions

        A = 1 + self.lam*(1 - self.beta)
        B = self.beta - self.lam*(1 - self.beta)
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  (tf.reduce_sum(y_pred**(self.beta+1), axis=1))/A - ((1+self.beta)/(A*B))*(sel_probs**B)
        sorted_losses = tf.sort(losses)
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_losses = sorted_losses[:k]  # Keep only k smallest residuals

        return tf.reduce_mean(trimmed_losses)

# TSCCE loss implementation
class TSCCE(Loss):
    def __init__(self, trim_ratio=0.2):
        super().__init__()
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        # Clip predictions to avoid log(0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        log_probs = tf.math.log(y_pred)
        # Get log probability of the correct class for each sample
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        per_sample_loss = -tf.gather_nd(log_probs, indices)
        # Trim top X% highest-loss samples
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_values, _ = tf.math.top_k(-per_sample_loss, k=k, sorted=False)
        trimmed_loss = -tf.reduce_mean(trimmed_values)

        return trimmed_loss

# DPD loss implementation
class TDPDSCCE(Loss):
    def __init__(self, beta, trim_ratio): # beta is the DPD tuning parameter. set trim_ratio to 0 to disable trimming, as done in our paper
        super().__init__()
        self.beta = float(beta)
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions

        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  tf.reduce_sum(y_pred**(self.beta+1), axis=1) - (1+1/self.beta)*(sel_probs**self.beta)
        sorted_losses = tf.sort(losses)
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_losses = sorted_losses[:k]  # Keep only k smallest residuals

        return tf.reduce_mean(trimmed_losses)

# SCE loss implementation
class SCE(Loss):
    def __init__(self, alpha, beta):
        super().__init__()
        self.alpha = float(alpha)
        self.beta = float(beta)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0) # Clip predictions

        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses = -self.alpha*tf.math.log(sel_probs) + self.beta*6*(1 - sel_probs)

        return tf.reduce_mean(losses)

# GCE loss implementation
class GCE(Loss):
    def __init__(self, q):
        super().__init__()
        self.q = float(q)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        # Clip predictions
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)

        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  (1 - (sel_probs**self.q))/self.q

        return tf.reduce_mean(losses)

# RKLD loss implementation
class RKLD(Loss):
    def __init__(self):
        super().__init__()

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions

        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)

        losses =  tf.reduce_sum(y_pred*tf.math.log(y_pred), axis=1) + 2*(1 - sel_probs)
        return tf.reduce_mean(losses)

# FCL loss implementation
class FCL(Loss):
    def __init__(self, mu):
        super().__init__()
        self.mu = float(mu)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32) # Clip predictions
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)

        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =   (-tf.math.log(sel_probs))**(1-self.mu)/tf.exp(tf.math.lgamma(tf.constant(2-self.mu))) + 2*(1-sel_probs)

        return tf.reduce_mean(losses)

# Define your NN model architecture

In [ ]:

# ── PART 1: Vanilla Transformer Architecture ──────────────────────────────────

#   • Identical call signature: get_model() → keras.Model with softmax output.
#   • All downstream training cells (rSDNet, CCE, DPD, GCE, SCE, FCL, RKLD,
#     MAE, TSCCE) call model.compile(…) and model.fit(…) completely unchanged.
#   • Falls back to globals IMG_H / IMG_W / IN_CHANNELS / NUM_CLASSES (set in
#     the dataset cell above) when called with no arguments.
#
# Architecture: image → PatchEmbed → PositionalEmbed → N × [Pre-LN MSA + FFN]
#               → GlobalAveragePool → LayerNorm → Dense(softmax)
# ─────────────────────────────────────────────────────────────────────────────

# Transformer hyper-parameters
# (Paper uses Adam + 250 epochs — those remain unchanged in all training cells.)
T_PATCH_SIZE = 8    # 8×8 patches → (32/8)² = 16 patches per image
T_D_MODEL    = 64   # token / embedding dimension
T_NUM_HEADS  = 4    # attention heads  (T_D_MODEL must be divisible by T_NUM_HEADS)
T_FFN_DIM    = 128  # feed-forward inner width (≈ 2× T_D_MODEL)
T_NUM_LAYERS = 4    # stacked Transformer encoder blocks
T_DROPOUT    = 0.1  # dropout rate applied in MSA + FFN paths


# ── Layer 1: Patch Embedding ──────────────────────────────────────────────────
class PatchEmbedding(keras.layers.Layer):
    """Splits image into non-overlapping p×p patches and linearly projects each."""
    def __init__(self, patch_size, d_model, **kw):
        super().__init__(**kw)
        self.patch_size = patch_size
        self.proj = keras.layers.Dense(
            d_model, kernel_initializer=GlorotUniform(seed=42))

    def call(self, x):
        p    = self.patch_size
        H, W, C = x.shape[1], x.shape[2], x.shape[3]
        # extract_patches → (B, H/p, W/p, p*p*C)
        patches = tf.image.extract_patches(
            x, sizes=[1, p, p, 1], strides=[1, p, p, 1],
            rates=[1, 1, 1, 1], padding='VALID')
        n_patches = (H // p) * (W // p)
        patches   = tf.reshape(patches, [tf.shape(x)[0], n_patches, p * p * C])
        return self.proj(patches)                  # (B, n_patches, d_model)


# ── Layer 2: Transformer Encoder Block ───────────────────────────────────────
class TransformerEncoderBlock(keras.layers.Layer):
    """Pre-LN Transformer: LayerNorm → MSA → Residual → LayerNorm → FFN → Residual."""
    def __init__(self, d_model, num_heads, ffn_dim, dropout, **kw):
        super().__init__(**kw)
        self.norm1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.msa   = keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads,
            kernel_initializer=GlorotUniform(seed=42))
        self.drop1 = keras.layers.Dropout(dropout)

        self.norm2 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.ffn   = keras.Sequential([
            keras.layers.Dense(ffn_dim, activation='gelu',
                               kernel_initializer=GlorotUniform(seed=42)),
            keras.layers.Dense(d_model,
                               kernel_initializer=GlorotUniform(seed=42)),
        ])
        self.drop2 = keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        # Pre-LayerNorm multi-head self-attention + residual
        h = self.norm1(x)
        h = self.msa(h, h, training=training)
        x = x + self.drop1(h, training=training)
        # Pre-LayerNorm feed-forward network + residual
        h = self.norm2(x)
        h = self.ffn(h)
        return x + self.drop2(h, training=training)


# ── Model Factory ─────────────────────────────────────────────────────────────
def get_model(input_shape=None, num_classes=None,
              patch_size=T_PATCH_SIZE, d_model=T_D_MODEL,
              num_heads=T_NUM_HEADS,   ffn_dim=T_FFN_DIM,
              num_layers=T_NUM_LAYERS, dropout=T_DROPOUT):
    """
    Vanilla Vision Transformer — drop-in for CNN get_model() in rSDNet.ipynb.
    Uses globals IMG_H / IMG_W / IN_CHANNELS / NUM_CLASSES when called with
    no arguments (default path used by ALL rSDNet training cells).
    """
    if input_shape is None:
        input_shape = (IMG_H, IMG_W, IN_CHANNELS)
    if num_classes is None:
        num_classes = NUM_CLASSES

    H, W, C   = input_shape
    n_patches = (H // patch_size) * (W // patch_size)

    inp = keras.Input(shape=input_shape, name='image')

    # 1. Linear patch embedding  →  (B, n_patches, d_model)
    x   = PatchEmbedding(patch_size, d_model, name='patch_embed')(inp)

    # 2. Learnable 1-D positional embeddings (one per patch)
    pos = keras.layers.Embedding(
        n_patches, d_model,
        embeddings_initializer=GlorotUniform(seed=42),
        name='pos_embed')(tf.range(n_patches))
    x   = x + pos                              # broadcast: (B, n_patches, d_model)

    # 3. Stack of vanilla Transformer encoder blocks
    for i in range(num_layers):
        x = TransformerEncoderBlock(
            d_model, num_heads, ffn_dim, dropout,
            name=f'trans_block_{i}')(x)

    # 4. Global average pool + final LayerNorm
    x   = keras.layers.GlobalAveragePooling1D(name='gap')(x)
    x   = keras.layers.LayerNormalization(epsilon=1e-6, name='out_norm')(x)

    # 5. Softmax classification head
    out = keras.layers.Dense(
        num_classes, activation='softmax',
        kernel_initializer=GlorotUniform(seed=42),
        name='classifier')(x)

    return keras.Model(inp, out, name='VanillaTransformer_rSDNet')


# rSDNet $(\beta, \lambda)$

In [ ]:
n_epochs = 250 # You can change the number of epochs here.
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=SDIV(beta=0.05, lam=-0.8, trim_ratio=0.0)) # You can chang beta and lambda here. Set trim_ratio to 0 to get the untrimmed version of the SDIV loss, as done in our paper
model.fit(X_train, y_train, epochs=n_epochs, batch_size=256, verbose=0, callbacks=[TQDMProgressCallback(desc="rSDNet (SDIV)")])
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy


# CCE

In [ ]:
n_epochs = 250 # You can change the number of epochs here.
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit(X_train, y_train, epochs=n_epochs, batch_size=256, verbose=0, callbacks=[TQDMProgressCallback(desc="CCE")])
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy


# TSCCE

In [ ]:
n_epochs = 250 # You can change the number of epochs for training
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=TSCCE(trim_ratio=0.2)) # You can change the trim_ratio
model.fit(X_train, y_train, epochs=n_epochs, batch_size=256, verbose=0, callbacks=[TQDMProgressCallback(desc="TSCCE")])
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy


# GCE

In [ ]:
n_epochs = 250 # You can change the number of epochs
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=GCE(q=0.7)) # You can change the q value
model.fit(X_train, y_train, epochs=n_epochs, batch_size=256, verbose=0, callbacks=[TQDMProgressCallback(desc="GCE")])
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy


# SCE

In [ ]:
n_epochs = 250 # You can change the number of epochs
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=SCE(alpha=0.5, beta=1.0)) # You can change the alpha and beta values
model.fit(X_train, y_train, epochs=n_epochs, batch_size=256, verbose=0, callbacks=[TQDMProgressCallback(desc="SCE")])
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy


# FCL

In [ ]:
n_epochs = 250 # You can change the number of epochs
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=FCL(mu=0.5)) # You can change the mu value
model.fit(X_train, y_train, epochs=n_epochs, batch_size=256, verbose=0, callbacks=[TQDMProgressCallback(desc="FCL")])
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy


# TDPD

In [ ]:
n_epochs = 250 # You can change the number of epochs as needed
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=TDPDSCCE(beta=0.5, trim_ratio=0)) # You can change the beta here. Set trim_ratio to 0 to get the untrimmed version of the DPD loss, as done in our paper
model.fit(X_train, y_train, epochs=n_epochs, batch_size=256, verbose=0, callbacks=[TQDMProgressCallback(desc="TDPD")])
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy


# MAE

In [ ]:
n_epochs = 250 # You can change the number of epochs as needed
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
y_train_corrupted = to_categorical(y_train, 10)
model.compile(optimizer='adam', loss='mae')
model.fit(X_train, y_train_corrupted, epochs=n_epochs, batch_size=256, verbose=0, callbacks=[TQDMProgressCallback(desc="MAE")])
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy


# RKLD

In [ ]:
n_epochs = 250 # You can change the number of epochs as needed
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=RKLD())
model.fit(X_train, y_train, epochs=n_epochs, batch_size=256, verbose=0, callbacks=[TQDMProgressCallback(desc="RKLD")])
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy
